In [22]:
import pandas as pd
import numpy as np
import re
import plotly.express as px


In [23]:
PROVINCES = {
    "buenos aires": ["buenos aires", "caba", "bs as", "capital federal"],
    "cordoba": ["cordoba", "córdoba"],
    "santa fe": ["santa fe", "rosario"],
    "mendoza": ["mendoza"],
    "tucuman": ["tucuman", "tucumán"],
    "entre rios": ["entre rios", "entre ríos"],
    "salta": ["salta"],
    "misiones": ["misiones"],
    "chaco": ["chaco"],
    "corrientes": ["corrientes"],
    "san juan": ["san juan"],
    "san luis": ["san luis", "villa mercedes"],
    "la pampa": ["la pampa"],
    "neuquen": ["neuquen", "neuquén"],
    "rio negro": ["rio negro", "río negro"],
    "chubut": ["chubut"],
    "santa cruz": ["santa cruz"],
    "tierra del fuego": ["tierra del fuego"],
    "jujuy": ["jujuy"],
    "formosa": ["formosa"],
    "la rioja": ["la rioja"],
    "catamarca": ["catamarca"],
    "santiago del estero": ["santiago del estero"],
}

PROVINCE_COORDS = {
    "buenos aires": (-34.61, -58.38),
    "cordoba": (-31.42, -64.18),
    "santa fe": (-31.63, -60.70),
    "mendoza": (-32.89, -68.84),
    "tucuman": (-26.83, -65.22),
    "entre rios": (-31.73, -60.53),
    "salta": (-24.79, -65.41),
    "misiones": (-27.36, -55.90),
    "chaco": (-27.45, -58.99),
    "corrientes": (-27.48, -58.83),
    "san juan": (-31.54, -68.52),
    "san luis": (-33.30, -66.34),
    "la pampa": (-36.62, -64.29),
    "neuquen": (-38.95, -68.06),
    "rio negro": (-40.81, -63.00),
    "chubut": (-43.30, -65.10),
    "santa cruz": (-51.62, -69.22),
    "tierra del fuego": (-54.80, -68.30),
    "jujuy": (-24.19, -65.30),
    "formosa": (-26.18, -58.18),
    "la rioja": (-29.41, -66.85),
    "catamarca": (-28.47, -65.78),
    "santiago del estero": (-27.78, -64.26),
}

In [24]:
def normalize_to_province(loc: str) -> str:
    if pd.isna(loc):
        return None

    loc = loc.lower()

    # limpiar ruido
    loc = re.sub(r"[^a-záéíóúñ\s]", " ", loc)
    loc = re.sub(r"\s+", " ", loc)

    # detectar remoto
    if any(x in loc for x in ["remote", "remoto", "home office"]):
        return "remote"

    # buscar provincia
    for province, keywords in PROVINCES.items():
        for keyword in keywords:
            if keyword in loc:
                return province

    return loc

In [25]:
import os
import glob

def load_and_concat_parquet(base_path: str) -> pd.DataFrame:
    """
    Recorre recursivamente el directorio base_path,
    carga todos los archivos .parquet encontrados
    y devuelve un único DataFrame concatenado.
    """
    # Buscar todos los archivos parquet dentro del path
    parquet_files = glob.glob(os.path.join(base_path, "**", "*.parquet"), recursive=True)

    if not parquet_files:
        raise FileNotFoundError(f"No se encontraron archivos parquet en {base_path}")

    # Leer cada archivo y acumular en una lista
    dfs = []
    for file in parquet_files:
        try:
            df = pd.read_parquet(file)
            dfs.append(df)
        except Exception as e:
            print(f"Error leyendo {file}: {e}")

    # Concatenar todos los DataFrames
    if dfs:
        return pd.concat(dfs, ignore_index=True)
    else:
        raise ValueError("No se pudo cargar ningún DataFrame válido.")

In [26]:
raw_df = load_and_concat_parquet(r"..\\data\raw")
print(raw_df.shape)

(1179, 20)


In [27]:
raw_df.head()

,title,company,location,salary,salary_currency,job_type,modality,experience_level,posted_date,url,description_raw,tags,skills,source_site,applicants,source,ds,scraped_at,job_key,title_lower
0,Product Analysten\nWherex,Wherex,Santiago,NaN,NaN,Full time,Híbrido,Semi Senior,2026-02-12T21:16:52+00:00,https://www.getonbrd.com/empleos/innovacion-ag...,Wherex es una plataforma web de licitaciones i...,"[Python, Data Analysis, SQL, Power BI, Data Vi...",[],get_on_board,74,getonboard,2026-04-20,2026-04-20T22:32:56.479675,11033c0b6e2c1fb12c27,product analysten\nwherex
1,Data Scientisten\nArtefact Latam,Artefact LatAm,NaN,1500 - 2000,USD/MONTH,Full time,Remoto,Semi Senior,2025-10-01T18:14:12+00:00,https://www.getonbrd.com/empleos/data-science-...,"Somos Artefact, una consultora líder a nivel m...","[Python, Analytics, Git, Data Analysis, SQL, A...",[],get_on_board,706,getonboard,2026-04-20,2026-04-20T22:32:56.479675,ffe0694c36469217ca63,data scientisten\nartefact latam
2,Qa Engineer Ii (L4)En\nOpenloop,OpenLoop,Lima,NaN,NaN,Full time,Híbrido,Semi Senior,2026-03-25T22:46:48+00:00,https://www.getonbrd.com/jobs/sysadmin-devops-...,"About OpenLoopOpenLoop was co-founded by CEO, ...","[JavaScript, Python, QA, Virtualization, Amazo...",[],get_on_board,12,getonboard,2026-04-20,2026-04-20T22:32:56.479675,50a0f618d4b38a64362d,qa engineer ii (l4)en\nopenloop
3,Cloud Data Engineeren\nWiti,WiTi,Santiago,NaN,NaN,Full time,Híbrido,Semi Senior,2026-04-06T20:45:32+00:00,https://www.getonbrd.com/empleos/data-science-...,WiTi conecta talento tecnológico con proyectos...,"[Python, SQL, ETL, CI/CD, Infrastructure as Co...",[],get_on_board,12,getonboard,2026-04-20,2026-04-20T22:32:56.479675,862e0b5651bc898e812f,cloud data engineeren\nwiti
4,Analista De Business Intelligence Finanzasen\n...,Equifax Chile,Lima,NaN,NaN,Full time,Híbrido,Semi Senior,2026-04-09T23:48:20+00:00,https://www.getonbrd.com/jobs/data-science-ana...,En Equifax Perú transformamos datos en oportun...,"[Excel, Power BI, Automation, AI, Budgeting, G...",[],get_on_board,16,getonboard,2026-04-20,2026-04-20T22:32:56.479675,4c148f987a9516b964ad,analista de business intelligence finanzasen\n...


In [28]:
raw_df.isna().sum()

title                 0
company             195
location             92
salary              769
salary_currency     146
job_type              2
modality              0
experience_level      1
posted_date           1
url                   0
description_raw       0
tags                  0
skills              622
source_site           0
applicants          933
source                0
ds                    0
scraped_at            0
job_key               0
title_lower           0
dtype: int64

In [29]:
raw_df["province"] = raw_df["location"].apply(normalize_to_province)

In [30]:
raw_df["province"].value_counts()

province
buenos aires        659
santiago            117
santa fe             69
cordoba              50
entre rios           27
misiones             26
corrientes           15
tucuman              13
salta                11
ciudad de méxico      9
neuquen               9
chubut                9
lima                  8
la rioja              8
jujuy                 7
mendoza               7
bogotá                6
rio negro             6
chaco                 5
san luis              4
santa cruz            4
catamarca             3
guayaquil             2
valparaíso            2
iquique               2
montevideo            2
mcallen texas         1
cali                  1
puerto varas          1
quito                 1
bogotá o              1
antofagasta           1
san juan              1
Name: count, dtype: int64

In [87]:
df_counts = (
    raw_df[raw_df["province"] != "remote"]
    .groupby("province")
    .size()
    .reset_index(name="count")
)

In [88]:
df_counts["lat"] = df_counts["province"].map(lambda x: PROVINCE_COORDS.get(x, (None, None))[0])
df_counts["lon"] = df_counts["province"].map(lambda x: PROVINCE_COORDS.get(x, (None, None))[1])

In [89]:
# escala log para evitar círculos gigantes
# df_counts["size"] = np.log1p(df_counts["count"]) * 10
df_counts["size"] = df_counts["count"]

In [90]:
import plotly.io as pio
pio.renderers.default = "browser"

fig = px.scatter_map(
    df_counts,
    lat="lat",
    lon="lon",
    size="size",              # 👈 diámetro proporcional
    hover_name="province",
    hover_data={"count": True, "lat": False, "lon": False},
    zoom=3,
    center={"lat": -38, "lon": -63},
    map_style="open-street-map",
    title="Demanda de empleos por provincia (Argentina)"
)

fig.show()

In [35]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1179 entries, 0 to 1178
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   title             1179 non-null   str   
 1   company           984 non-null    str   
 2   location          1087 non-null   str   
 3   salary            410 non-null    object
 4   salary_currency   1033 non-null   str   
 5   job_type          1177 non-null   str   
 6   modality          1179 non-null   str   
 7   experience_level  1178 non-null   str   
 8   posted_date       1178 non-null   str   
 9   url               1179 non-null   str   
 10  description_raw   1179 non-null   str   
 11  tags              1179 non-null   object
 12  skills            557 non-null    object
 13  source_site       1179 non-null   str   
 14  applicants        246 non-null    object
 15  source            1179 non-null   str   
 16  ds                1179 non-null   str   
 17  scraped_at        1179 no

In [37]:
raw_df.loc[raw_df["location"]=="san luis"]

,title,company,location,salary,salary_currency,job_type,modality,experience_level,posted_date,url,...,tags,skills,source_site,applicants,source,ds,scraped_at,job_key,title_lower,province


In [46]:
raw_df.loc[
    raw_df["province"].str.lower().str.contains("san luis", na=False)
]

,title,company,location,salary,salary_currency,job_type,modality,experience_level,posted_date,url,...,tags,skills,source_site,applicants,source,ds,scraped_at,job_key,title_lower,province
257,Vendedor/A,Portal Empleo,SAN LUIS - VILLA MERCEDES - VILLA MERCEDES - R...,A Convenir,$ARS,Tiempo parcial,Otros,5 años,27/02/2026,https://portalempleo.gob.ar/OfertasLaborales/D...,...,"[Vacantes, Disponibilidad horaria, Salario, Re...","[Primarios, Secundarios]",portal_empleo,None,portal_empleo,2026-04-22,2026-04-22T22:16:11.194477,3625a0d4b5e1d8e4e00a,vendedor/a,san luis
308,Vendedor/A De Comidas Rápidas,Portal Empleo,SAN LUIS - SAN LUIS - - 25 de Mayo 150 D5700,A Convenir,$ARS,Tiempo parcial,Otros,No,12/03/2026,https://portalempleo.gob.ar/OfertasLaborales/D...,...,"[Vacantes, Disponibilidad horaria, Salario, Re...",[],portal_empleo,None,portal_empleo,2026-04-22,2026-04-22T22:16:11.194477,5c75f325a5aecc8a082a,vendedor/a de comidas rápidas,san luis
345,Vendedor/A De Menaje O Bazar,Portal Empleo,SAN LUIS - SAN LUIS - SAN LUIS - SAN MARTIN 633,A Convenir,$ARS,Tiempo parcial,Otros,No,02/03/2026,https://portalempleo.gob.ar/OfertasLaborales/D...,...,"[Vacantes, Disponibilidad horaria, Salario, Re...",[],portal_empleo,None,portal_empleo,2026-04-22,2026-04-22T22:16:11.194477,790cf5998ea80e4638ca,vendedor/a de menaje o bazar,san luis
419,Auxiliar De Grupo Logístico,Portal Empleo,SAN LUIS - VILLA MERCEDES - VILLA MERCEDES - E...,A Convenir,$ARS,Tiempo parcial,Otros,No,28/02/2026,https://portalempleo.gob.ar/OfertasLaborales/D...,...,"[Vacantes, Disponibilidad horaria, Salario, Re...","[Secundarios, versión:0.7.12+f2612ee]",portal_empleo,None,portal_empleo,2026-04-22,2026-04-22T22:16:11.194477,2676f95982eb42802de5,auxiliar de grupo logístico,san luis


In [57]:
# Lista de preposiciones comunes en español
stopwords = [
    # Preposiciones
    "a","ante","bajo","cabe","con","contra","de","desde","durante","en","entre",
    "hacia","hasta","mediante","para","por","según","sin","so","sobre","tras",
    # Conectores
    "y","e","ni","o","u","pero","mas","aunque","sino","que","como","porque",
    "pues","entonces","luego","ya","ya que","dado","dado que","mientras","cuando",
    "donde","si","siempre","siempre que","a fin","a fin de","con el objeto de",
    # Artículos/determinantes
    "el","la","los","las","un","una","unos","unas","este","esta","estos","estas",
    "ese","esa","esos","esas","aquel","aquella","aquellos","aquellas",
    # Pronombres
    "yo","tú","él","ella","nosotros","ustedes","ellos","quien","quienes","cuyo","cuya",
    # Adverbios frecuentes
    "muy","más","menos","también","solo","incluso","además","apenas"
]

In [55]:
# 1. Pasar todo a minúsculas y limpiar signos
words = raw_df["title"].str.lower().str.replace(r"[^\w\s]", "", regex=True).str.split()

# 2. Explode para tener una fila por palabra
words_exploded = words.explode()

# 3. Filtrar quitando stopwords
words_filtered = words_exploded[~words_exploded.isin(stopwords)]

# 4. Contar frecuencia
freq = words_filtered.value_counts()

print(freq[:10])

title
analista       76
vendedora      63
remoto         58
data           54
atencin        53
trabajadora    52
comercial      52
mostrador      47
tecnologa      46
tcnico         46
Name: count, dtype: int64


In [83]:
# Función para tomar las primeras N palabras de cada título
def primeras_palabras(texto, n=3):
    palabras = texto.lower().split()
    return " ".join(palabras[:n])  # solo las primeras n

# Aplicar la función a la columna
jobs = raw_df["title"].apply(lambda x: primeras_palabras(x, n=5))

# Contar frecuencia de esas secuencias
freq = jobs.value_counts()

print(freq[:10])
print("===========================================")
print("Empleos totales : ",freq[:10].sum())


title
trabajador/a de atención de mostrador    46
vendedor/a                               17
operario/a de línea de montaje           13
auxiliar administrativo                  11
jefa zonal de ventas para                11
repositor/a                              10
ayudante de ventas de comercio            9
ayudante de cocina o peón                 8
peón de producción de frigorífico         8
cajero/a de híper o supermercado          7
Name: count, dtype: int64
Empleos totales :  140


In [86]:
print(freq[:10]/freq[:10].sum())
print("===========================================")
print("Empleos totales : ",freq[:10].sum())

title
trabajador/a de atención de mostrador    0.328571
vendedor/a                               0.121429
operario/a de línea de montaje           0.092857
auxiliar administrativo                  0.078571
jefa zonal de ventas para                0.078571
repositor/a                              0.071429
ayudante de ventas de comercio           0.064286
ayudante de cocina o peón                0.057143
peón de producción de frigorífico        0.057143
cajero/a de híper o supermercado         0.050000
Name: count, dtype: float64
Empleos totales :  140


In [77]:
raw_df.loc[
    raw_df["title"].str.lower().str.contains("data", na=False)
]

,title,company,location,salary,salary_currency,job_type,modality,experience_level,posted_date,url,...,tags,skills,source_site,applicants,source,ds,scraped_at,job_key,title_lower,province
1,Data Scientisten\nArtefact Latam,Artefact LatAm,NaN,1500 - 2000,USD/MONTH,Full time,Remoto,Semi Senior,2025-10-01T18:14:12+00:00,https://www.getonbrd.com/empleos/data-science-...,...,"[Python, Analytics, Git, Data Analysis, SQL, A...",[],get_on_board,706,getonboard,2026-04-20,2026-04-20T22:32:56.479675,ffe0694c36469217ca63,data scientisten\nartefact latam,NaN
3,Cloud Data Engineeren\nWiti,WiTi,Santiago,NaN,NaN,Full time,Híbrido,Semi Senior,2026-04-06T20:45:32+00:00,https://www.getonbrd.com/empleos/data-science-...,...,"[Python, SQL, ETL, CI/CD, Infrastructure as Co...",[],get_on_board,12,getonboard,2026-04-20,2026-04-20T22:32:56.479675,862e0b5651bc898e812f,cloud data engineeren\nwiti,santiago
5,Senior Data Consultanten\nArtefact Latam,Artefact LatAm,NaN,2500 - 3200,USD/MONTH,Full time,Remoto,Senior,2025-11-06T15:09:31+00:00,https://www.getonbrd.com/empleos/data-science-...,...,"[Python, Data Analysis, English, Data Visualiz...",[],get_on_board,130,getonboard,2026-04-20,2026-04-20T22:32:56.479675,7f5a16ead88784ae1ef2,senior data consultanten\nartefact latam,NaN
6,Big Data & Reporting Leaden\nCoderslab.Io,Coderslab.io,NaN,3500 - 3700,USD/MONTH,Full time,Remoto,Senior,2026-03-11T16:04:48+00:00,https://www.getonbrd.com/jobs/data-science-ana...,...,"[Java, Python, Analytics, PostgreSQL, REST API...",[],get_on_board,57,getonboard,2026-04-20,2026-04-20T22:32:56.479675,6bdaabc399bd8f3c4efd,big data & reporting leaden\ncoderslab.io,NaN
7,Data Analyst (Data Operations)En\nEquifax Chile,Equifax Chile,Santiago,NaN,NaN,Full time,Híbrido,Semi Senior,2026-04-15T13:52:55+00:00,https://www.getonbrd.com/jobs/data-science-ana...,...,"[Python, SQL, BigQuery, Scrum, API, Jira, Data...",[],get_on_board,42,getonboard,2026-04-20,2026-04-20T22:32:56.479675,a1bfdb8a5aaee0016757,data analyst (data operations)en\nequifax chile,santiago
9,Data Engineeren\nNeuralworks,NeuralWorks,Santiago,NaN,NaN,Full time,Híbrido,Semi Senior,2026-04-08T14:23:05+00:00,https://www.getonbrd.com/jobs/data-science-ana...,...,"[C, Python, SQL, Virtualization, Amazon Web Se...",[],get_on_board,94,getonboard,2026-04-20,2026-04-20T22:32:56.479675,75e65b99af306f3cf7d5,data engineeren\nneuralworks,santiago
14,Data Engineeren\nHaystack News,Haystack News,Lima,3100 - 4500,USD/MONTH,Full time,Híbrido,Semi Senior,2024-06-07T14:56:09+00:00,https://www.getonbrd.com/empleos/data-science-...,...,"[Python, SQL, Big Data, Data Warehouse, A/B Te...",[],get_on_board,286,getonboard,2026-04-20,2026-04-20T22:32:56.479675,0945a29742bf57c001d4,data engineeren\nhaystack news,lima
15,Data Analyst (Proyecto)En\nBc Tecnología,BC Tecnología,Santiago,NaN,NaN,Freelance,Híbrido,Semi Senior,2026-02-17T15:49:14+00:00,https://www.getonbrd.com/jobs/data-science-ana...,...,"[Python, Analytics, Agile, Data Analysis, SQL,...",[],get_on_board,46,getonboard,2026-04-20,2026-04-20T22:32:56.479675,2e867c7e869c8fe17e5d,data analyst (proyecto)en\nbc tecnología,santiago
16,Data Scientist / Machine Learning Engineeren\n...,CyD Tecnología,Santiago,NaN,NaN,Full time,Híbrido,Semi Senior,2026-04-08T14:47:07+00:00,https://www.getonbrd.com/empleos/data-science-...,...,"[Python, Agile, SQL, Erlang, Big Data, BigQuer...",[],get_on_board,137,getonboard,2026-04-20,2026-04-20T22:32:56.479675,618d40d5570f9bbde0a7,data scientist / machine learning engineeren\n...,santiago
17,Data Engineer – Proyecto (Híbrida)En\nBc Tecno...,BC Tecnología,Santiago,NaN,NaN,Full time,Híbrido,Semi Senior,2026-03-03T17:08:48+00:00,https://www.getonbrd.com/empleos/data-science-...,...,"[Python, PostgreSQL, SQL, Web server, DevOps, ...",[],get_on_board,51,getonboard,2026-04-20,2026-04-20T22:32:56.479675,f2b4875e3137f5c78b60,data engineer – proyecto (híbrida)en\nbc tecno...,santiago
